In [0]:
# Databricks notebook source

from __future__ import annotations
import uuid

dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

SCRIPT_NAME = "236_build_silver_dim_county.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.dim_project"
TARGET_TABLE = f"{catalog}.silver.dim_county"
LOG = f"{catalog}.staging.pipeline_run_log"

def log_run(status, row_count, message):
    spark.sql(f"""
        INSERT INTO {LOG}
        VALUES (
            '{RUN_ID}',
            '{SCRIPT_NAME}',
            '{status}',
            {row_count if row_count else 'NULL'},
            '{message}',
            current_timestamp()
        )
    """)

# ------------------------------------------------------------
# Build county dimension
# ------------------------------------------------------------
try:

    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH source_values AS (
        SELECT DISTINCT
            TRIM(county) AS source_county
        FROM {SOURCE_TABLE}
    )

    , source_normalized AS (
        SELECT
            CASE
                WHEN source_county IS NULL OR source_county = '' THEN 'UNKNOWN'
                ELSE source_county
            END AS source_county
        FROM source_values
    )

    , alias_map AS (
        SELECT * FROM VALUES
            ('De Witt','DeWitt')
        AS t(source_county, canonical_county)
    )

    , source_canonical AS (
        SELECT DISTINCT
            s.source_county,
            CASE
                WHEN s.source_county = 'UNKNOWN' THEN 'UNKNOWN'
                ELSE COALESCE(a.canonical_county, s.source_county)
            END AS county
        FROM source_normalized s
        LEFT JOIN alias_map a
            ON s.source_county = a.source_county
    )

    , distinct_values AS (
        SELECT DISTINCT county
        FROM source_canonical
    )

    , ordered_values AS (
        SELECT
            county,
            ROW_NUMBER() OVER (ORDER BY county) AS seq_nbr
        FROM distinct_values
        WHERE county <> 'UNKNOWN'
    )

    SELECT
        -1 AS county_key,
        'UNKNOWN' AS county,
        'Unknown County, Texas, United States' AS county_location,
        FALSE AS is_official_texas_county

    UNION ALL

    SELECT
        seq_nbr AS county_key,
        county,
        CONCAT(county, ' County, Texas, United States'),
        TRUE
    FROM ordered_values
    """)

    row_count = spark.sql(f"SELECT COUNT(*) c FROM {TARGET_TABLE}").collect()[0]["c"]

    log_run("SUCCESS", row_count, "Built dim_county")

    print("dim_county rows:", row_count)

except Exception as e:
    log_run("FAILED", None, str(e))
    raise